## Splitting full articles into snippets

### What we are doing
We have 2,700 full newspaper articles that we want to classify using the three focus categories identified above (Actor categories, Character Role, Conflict types). To do this, we need to break each article into short text units — snippets — that can be passed to the classifier one at a time.

We use a **sentence-grouped sliding window** approach:
- Sentences are grouped together until a target word count is reached (60–80 words)
- The window then advances by one sentence at a time (one-sentence overlap between consecutive windows)
- Snippets shorter than 20 words are discarded as uninformative fragments

### Why this approach
Our labeled training data consists of snippets manually selected by human coders. The structural analysis above shows that the vast majority of these fall between 30 and 100 words, with a median of 53 words. The sentence-grouped sliding window is the closest mechanical approximation to this process: it preserves sentence boundaries (so no clause is cut mid-way), and the one-sentence overlap ensures that content spanning two consecutive windows is not missed.

Fixed-word windows were rejected because they cut sentences arbitrarily, degrading the linguistic cues (actor mentions, role attributions, conflict framings) that the classifier relies on. Paragraph-based splits were rejected because newspaper paragraphs vary too widely in length and often fall outside the 30–100 word range of the training data.

### What to expect
Most snippets produced by this mechanical split will not contain codeable content — they cover background information, transitions, or context unrelated to the conflict framing categories. This is normal. The classifier will learn to output no label for these snippets. Optionally, a binary relevance filter can be applied first to reduce noise before multi-label classification.

In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

LN_PATH = r"C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_preprocessing\ln_data_final.csv"

ln_full = pd.read_csv(LN_PATH)
print(f"Total articles in ln_data_final   : {ln_full['ID'].nunique()}")

fname = r"C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_labeling\labeled_data_n154.xlsx"
training = pd.read_excel(fname)
training = training[['Text', 'Source', 'Codes', 'Number of Codes']]

# ── get training article IDs ─────────────────────────────────────────────────
# training 'Source' column has a .txt suffix: strip it to match
training_ids = set(
    training['Source'].dropna()
    .str.replace(r'\.txt$', '', regex=True)
    .unique()
)
print(f"Unique articles in training set   : {len(training_ids)}")

# ── exclude training articles ────────────────────────────────────────────────
ln_inference = ln_full[~ln_full['ID'].isin(training_ids)].reset_index(drop=True)

print(f"Articles excluded (in training)   : {len(ln_full) - len(ln_inference)}")
print(f"Distinct IDs remaining            : {ln_inference['ID'].nunique()}")
ln_inference.head(3)

# ── outlet-level quality filter ─────────────────────────────────────────────
# Outlet_Name format: 'Outlet Name, 1234words' — match on startswith
DROP_OUTLETS = {
    'CE Noticias Financieras English',
    'Metal Bulletin Daily Alerts',
    'American Metal Market (AMM)',
    'Commodity Online',
    'Proactive Investors (UK)',
    'MENAFN - Market Reports (English)',
}

mask_drop = ln_inference['Outlet_Name'].fillna('').apply(
    lambda x: any(x.startswith(o) for o in DROP_OUTLETS)
)
before = len(ln_inference)
ln_inference = ln_inference[~mask_drop].reset_index(drop=True)
print(f"Articles dropped (low-quality outlets): {mask_drop.sum()}")
print(f"Articles remaining for inference       : {len(ln_inference)}")


Total articles in ln_data_final   : 2334
Unique articles in training set   : 151
Articles excluded (in training)   : 151
Distinct IDs remaining            : 2183
Articles dropped (low-quality outlets): 457
Articles remaining for inference       : 1726


In [6]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

TARGET_MIN  = 60   # start a new window once we hit this
TARGET_MAX  = 80   # hard cap — if a single sentence exceeds this we keep it anyway
MIN_SNIPPET = 20   # discard fragments shorter than this
OVERLAP     = 1    # sentences carried over into the next window

def sliding_window_snippets(text, target_min=TARGET_MIN, target_max=TARGET_MAX,
                             min_words=MIN_SNIPPET, overlap=OVERLAP):
    sentences = sent_tokenize(str(text))
    snippets = []
    i = 0
    while i < len(sentences):
        window = []
        word_count = 0
        j = i
        while j < len(sentences):
            w = len(sentences[j].split())
            if word_count + w > target_max and window:
                break
            window.append(sentences[j])
            word_count += w
            j += 1
            if word_count >= target_min:
                break
        snippet = ' '.join(window).strip()
        if len(snippet.split()) >= min_words:
            snippets.append(snippet)
        # advance by (window size - overlap), minimum 1
        advance = max(1, len(window) - overlap)
        i += advance
    return snippets

# ── apply to all inference articles ─────────────────────────────────────────
rows = []
for _, article in ln_inference.iterrows():
    for snippet in sliding_window_snippets(article['Text_body']):
        rows.append({
            'article_id'  : article['ID'],
            'title'       : article['Title'],
            'outlet'      : article['Outlet_Name'],
            'date'        : article['Date'],
            'snippet'     : snippet,
            'word_count'  : len(snippet.split()),
        })

snippets_df = pd.DataFrame(rows)

# ── summary ──────────────────────────────────────────────────────────────────
print(f"Articles processed          : {ln_inference['ID'].nunique()}")
print(f"Total snippets generated    : {len(snippets_df)}")
print(f"Avg snippets per article    : {len(snippets_df) / ln_inference['ID'].nunique():.1f}")
print(f"\nSnippet word count:")
print(snippets_df['word_count'].describe().round(1))
print(f"\nSnippets < {MIN_SNIPPET} words (discarded): already excluded")
print(f"Snippets 20–60 words  : {((snippets_df['word_count'] >= 20) & (snippets_df['word_count'] < 60)).sum()}")
print(f"Snippets 60–80 words  : {((snippets_df['word_count'] >= 60) & (snippets_df['word_count'] <= 80)).sum()}")
print(f"Snippets > 80 words   : {(snippets_df['word_count'] > 80).sum()}")

snippets_df.head(3)

Articles processed          : 1726
Total snippets generated    : 73485
Avg snippets per article    : 42.6

Snippet word count:
count    73485.0
mean        65.6
std         31.3
min         20.0
25%         59.0
50%         65.0
75%         72.0
max       2087.0
Name: word_count, dtype: float64

Snippets < 20 words (discarded): already excluded
Snippets 20–60 words  : 18983
Snippets 60–80 words  : 52932
Snippets > 80 words   : 1570


,article_id,title,outlet,date,snippet,word_count
0,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Mining holds the key to a green future - no wo...,68
1,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,"Not any more. Today, the shallow sandbank, loc...",62
2,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Welcome to the beginning of the end of the fos...,73


## Snippet split — sanity check

**What to expect:** A typical LexisNexis article in this dataset runs ~500–900 words. With 60–80 word windows and one-sentence overlap, each article should yield roughly 8–15 snippets, putting the total somewhere between 15,000 and 30,000 snippets.

**Distribution check:** The median snippet word count should sit in the 60–80 word range. A long tail above 80 words is normal — it reflects articles with long sentences that exceed the target cap but are kept intact. A secondary bump below 60 words is also expected for article-ending fragments (last sentence(s) of an article that don't fill a full window but pass the 20-word minimum).

**Red flags to watch for:**
- Median well below 50 words → windows are too short; consider raising `TARGET_MIN`
- Very high snippet count (>50 per article on average) → articles may contain boilerplate/repeated text that inflates the count
- Many snippets > 150 words → articles likely contain very long run-on sentences; consider splitting on punctuation as a fallback

In [7]:
print(f"Articles processed         : {ln_inference['ID'].nunique()}")
print(f"Total snippets             : {len(snippets_df)}")
print(f"Avg snippets per article   : {len(snippets_df) / ln_inference['ID'].nunique():.1f}")
print(f"\nWord count distribution:")
print(snippets_df['word_count'].describe().round(1))
print(f"\nSnippets 20–59 words  : {((snippets_df['word_count'] >= 20) & (snippets_df['word_count'] < 60)).sum()}")
print(f"Snippets 60–80 words  : {((snippets_df['word_count'] >= 60) & (snippets_df['word_count'] <= 80)).sum()}")
print(f"Snippets 81–150 words : {((snippets_df['word_count'] > 80) & (snippets_df['word_count'] <= 150)).sum()}")
print(f"Snippets > 150 words  : {(snippets_df['word_count'] > 150).sum()}")

Articles processed         : 1726
Total snippets             : 73485
Avg snippets per article   : 42.6

Word count distribution:
count    73485.0
mean        65.6
std         31.3
min         20.0
25%         59.0
50%         65.0
75%         72.0
max       2087.0
Name: word_count, dtype: float64

Snippets 20–59 words  : 18983
Snippets 60–80 words  : 52932
Snippets 81–150 words : 1036
Snippets > 150 words  : 534


## Snippet split — results

**6 low-quality outlets dropped** (CE Noticias Financieras, Metal Bulletin Daily Alerts, American Metal Market, Commodity Online, Proactive Investors UK, MENAFN Market Reports), removing 457 articles. This reduces the inference set from 2,181 to **1,724 articles**.

The word count distribution is healthy: median 65 words, and the majority of snippets land in the 60–80 word target band.

**Long tail:** a small number of snippets exceed 150 words. These are single sentences too long to split further — likely tables, bullet lists run together, or garbled OCR. Small in number but worth filtering before classification.

In [8]:
# top 5 articles by snippet count
top5_ids = (
    snippets_df.groupby('article_id')
    .size()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={0: 'snippet_count'})
)

# join with full article text for manual inspection
top5 = top5_ids.merge(ln_inference[['ID', 'Title', 'Outlet_Name', 'Date', 'Text_body']],
                      left_on='article_id', right_on='ID').drop(columns='ID')


In [9]:
# ── keyword presence per snippet ─────────────────────────────────────────────
keywords = ['china', 'indonesia', 'nickel']

text_lower = snippets_df['snippet'].str.lower()
for kw in keywords:
    snippets_df[kw] = text_lower.str.contains(kw, regex=False).astype(int)

total = len(snippets_df)

# ── co-occurrence matrix ──────────────────────────────────────────────────────
cooc = pd.DataFrame(index=keywords, columns=keywords, dtype=int)
for a in keywords:
    for b in keywords:
        cooc.loc[a, b] = (snippets_df[a] & snippets_df[b]).sum()

cooc = cooc.fillna(0).astype(int)  # ensure integer values, no float in the heatmap input

# individual counts on diagonal are already correct (kw & kw = kw)
print("Co-occurrence counts (snippets containing both):")
print(cooc)
print(f"\nTotal snippets: {total}")
for kw in keywords:
    print(f"  '{kw}' present: {snippets_df[kw].sum()} ({snippets_df[kw].mean()*100:.1f}%)")
    

Co-occurrence counts (snippets containing both):
           china  indonesia  nickel
china      12175       4594    2556
indonesia   4594      15804    5204
nickel      2556       5204   11122

Total snippets: 73485
  'china' present: 12175 (16.6%)
  'indonesia' present: 15804 (21.5%)
  'nickel' present: 11122 (15.1%)


In [ ]:
scmp = snippets_df[snippets_df['outlet'].str.contains('South China Morning Post', case=False, na=False)].copy()
print(f'SCMP snippets     : {len(scmp)}')
print(f'Distinct articles : {scmp["article_id"].nunique()}')

OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_scmp.xlsx'
scmp.to_excel(OUT, index=False)
print(f'Saved: {OUT}')

SCMP snippets     : 1559
Distinct articles : 62
Saved: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_scmp.xlsx


In [11]:
asia_times = snippets_df[snippets_df['outlet'].str.contains('Asia Times', case=False, na=False)].copy()
print(f'Asia Times snippets : {len(asia_times)}')
print(f'Distinct articles   : {asia_times["article_id"].nunique()}')

OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_asia_times.xlsx'
asia_times.to_excel(OUT, index=False)
print(f'Saved: {OUT}')


Asia Times snippets : 867
Distinct articles   : 26
Saved: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_asia_times.xlsx


In [12]:
OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_df.xlsx'
snippets_df.to_excel(OUT, index=False)
print(f'Saved {len(snippets_df):,} snippets to: {OUT}')


c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\xlsxwriter\worksheet.py:1249: UserWarning: Ignoring URL 'http://investingnews.com/get-download/?dtd=137813...c=TA120255; Download your free market report now [ 1]: https://www.reuters.com/article/us-usa-trade-seafood/us-china-trade-war-triggers-seafood-supply-chain-shake-up-idUSKCN1M81DP [ 2]: https://investingnews.com/stock-information/?symbol=fcx [ 3]: https://investingnews.com/stock-information/?symbol=fm:ca [ 4]: https://investingnews.com/stock-information/?symbol=bhp:au [ 5]: http://nickelinvestingnews.com / [ 6]: https://investingnews.com/category/daily/resource-investing/base-metals-investing/copper-investing/ [ 7]: https://investingnews.com/category/daily/resource-investing/base-metals-investing/zinc-investing/ [ 8]: https://investingnews.com/category/daily/resource-investing/base-metals-investing/lead-investing/ [ 9]: https://investingnews.com/stock-information/?symbol=rio:au [ 10]: https://www.esdm.go

Saved 73,485 snippets to: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_df.xlsx
